## Lost in the Middle and Context Rot

Two related but distinct ways context quality degrades. "Lost in the Middle" is about WHERE information sits — position affects how well a model attends to it, even when nothing is technically missing. "Context Rot" is about WHAT ACCUMULATES over time — long-running conversations fill up with stale, low-value content that dilutes everything else.

## How it actually works

**Lost in the Middle — a measured, position-dependent attention bias:**
Research has consistently found that LLMs pay disproportionately MORE attention to
information positioned at the very START and very END of a long context, and
disproportionately LESS attention to information buried in the MIDDLE — even when
all of that information is equally relevant and equally present. This produces a
U-shaped recall curve across position, not a flat, even level of attention regardless
of where something sits.
Attention/Recall

high |\                              /|

 | \                            / |

     |  \                          /  |

         |   _/   |

low  |_________|

    Start        Middle           End

            Position in context

This is a SNAPSHOT problem — given a fixed set of information all present at once,
WHERE it sits changes how well it gets used.

**Context Rot — a TIME and ACCUMULATION problem, genuinely different:**
As a conversation or agent run goes on longer — more tool calls, more retries, more
intermediate steps — the context window accumulates increasing amounts of LOW-VALUE
or STALE content: failed attempts, superseded observations, verbose reasoning that
is no longer relevant. The signal-to-noise ratio of the context progressively
degrades over the course of a session, even though nothing was technically "lost in
the middle" — it is all still there, just increasingly diluted by accumulated content
that no longer matters.

**The clean way to hold both together:**
Lost in the Middle says: even GOOD information can be poorly placed.
Context Rot says: context fills with increasingly BAD information over time, regardless
of placement.
Both can be happening simultaneously in a single long-running session — relevant
information badly positioned, sitting inside an increasingly noisy, junk-filled context.

**A concrete example of context rot specifically:**
An agent tries a tool call, gets an error, retries with a tweak, gets another error,
tries a third approach, finally succeeds. All three attempts — including the two
failed ones with their error messages — remain in context for every subsequent step
of the conversation, even though only the successful third attempt's result is still
actually relevant. After many more steps, the context contains a growing pile of
resolved errors and abandoned approaches that no longer matter, but still consume
context budget and dilute attention away from what is currently relevant.

## Where it breaks or gets dangerous

**Naively ordering RAG chunks by retrieval score can trigger Lost in the Middle directly:**
If you retrieve 10 chunks and simply inject them in score order, your single BEST
chunk might land in the middle of that list purely by virtue of list length — not
because of anything wrong with the chunk itself, but because of where it happened
to fall positionally. The fix is deliberate: place the highest-confidence result at
the position the model attends to best (start or end), not wherever raw sorting
happens to put it.

**Context rot is specifically a risk for LONG-RUNNING conversational or agentic
systems, not short pipelines:**
A system like your Personal Brand Autopilot — five sequential agent calls, then done —
is largely protected from context rot WITHIN one generation, because each agent
receives a relatively fresh, focused prompt rather than one accumulating conversation
across many turns. The genuine risk areas are systems designed for EXTENDED sessions:
your Local RAG Assistant if used in a long multi-turn chat, and your Finance Tracker's
AI chat assistant, which currently has no formal strategy (summarisation, sliding
window, or external memory — covered in earlier sessions) for managing a growing
conversation history at all.

**The mitigation strategies for context rot also need to actively DISCARD, not just compress:**
A naive summarisation strategy that compresses everything in a long conversation
equally — including the resolved errors and abandoned approaches — still carries
some of that low-value content forward in compressed form. A better strategy
specifically distinguishes between "this is resolved and no longer relevant, drop it
entirely" versus "this is still relevant, compress it but keep it."

**These two problems compound, they do not cancel each other out:**
A long-running system can simultaneously have relevant information poorly positioned
AND have accumulated significant junk content diluting everything — meaning the
fixes for one problem (e.g., reordering for position) do not automatically solve
the other (removing accumulated low-value content over time). Both need deliberate,
separate handling.

In [ ]:
# Lost in the Middle and Context Rot — illustrating both effects conceptually

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

key_fact = "The approval threshold for ACME Corp is exactly $25,000."

filler_facts = [
    "ACME Corp was founded in 2018.",
    "ACME Corp's main office is in the northern district.",
    "ACME Corp supplies raw materials for manufacturing.",
    "ACME Corp has 340 employees globally.",
    "ACME Corp's primary contact changed last year.",
    "ACME Corp ships internationally on a quarterly basis.",
]

def build_context(key_fact_position):
    """Builds a context block with the key fact placed at start, middle, or end."""
    facts = filler_facts.copy()
    if key_fact_position == "start":
        return "\n".join([key_fact] + facts)
    elif key_fact_position == "middle":
        mid = len(facts) // 2
        return "\n".join(facts[:mid] + [key_fact] + facts[mid:])
    elif key_fact_position == "end":
        return "\n".join(facts + [key_fact])

question = "What is the approval threshold for ACME Corp?"

print("=== Lost in the Middle: same key fact, three different positions ===\n")

for position in ["start", "middle", "end"]:
    context = build_context(position)
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=40,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]
    )
    print(f"Key fact at: {position.upper()}")
    print(f"  Answer: {response.content[0].text.strip()}")
    print()

print("With only 6 filler facts, all three positions will likely still answer correctly —")
print("the effect becomes measurable at much larger context sizes (dozens of chunks,")
print("thousands of tokens), which is why this matters far more in production RAG")
print("systems retrieving many chunks than in a small illustrative example like this one.")
print()
print("=== Context Rot: conceptual illustration (not runnable at small scale) ===")
print("Imagine an agent's conversation after 15 tool calls, 4 of which failed and retried:")
print("- 4 failed attempts with error messages: still sitting in context, no longer relevant")
print("- 11 successful steps: mix of currently relevant and now-superseded information")
print("- The CURRENT actually relevant fact: a small fraction of the total accumulated text")
print()
print("Even with perfect positioning (Lost in the Middle solved), this accumulated")
print("low-value content still dilutes the signal — a genuinely separate problem.")

## My D365 analogy

Lost in the Middle is similar to a long, detailed audit document where the one
genuinely critical finding is sitting on page 14 of 20, surrounded by routine,
unremarkable observations. Reviewers reliably catch findings on page 1 and page 20 —
the executive summary and the final conclusion — but a critical finding buried in
the unremarkable middle pages gets meaningfully less attention, purely because of
where it physically sits in the document, not because of how important it actually is.

Context rot is a different, separate problem: a long-running support ticket thread
that has accumulated 40 messages over three weeks, where half of those messages are
now-resolved back-and-forth about a misunderstanding that got cleared up early on.
That resolved content is still sitting there in the thread, still consuming the
reader's attention and scrolling effort, even though it stopped being relevant
weeks ago. Nobody removed it. It just accumulated.

Both problems are real, and they are different. A well-organised audit report still
needs someone to actually clear out the resolved, no-longer-relevant noise from a
long-running thread — fixing one problem does not fix the other.

## LinkedIn post idea

Two AI systems can have the exact same information available, and still perform
differently — purely because of where that information sits, or how much
accumulated clutter surrounds it.

The first problem is positional. Models pay more attention to information at the
start and end of a long context than information buried in the middle, even when
everything present is equally relevant. The fix is deliberate placement, not more
content.

The second problem is accumulation. A long-running AI conversation fills up over
time with resolved errors, abandoned attempts, and superseded information that
nobody actively removes. Nothing is technically lost. It is just increasingly
diluted by content that stopped mattering several steps ago.

Both problems sound abstract until you have sat through a 40-message support thread
where half the content is a resolved misunderstanding from three weeks ago, still
sitting there, still consuming everyone's attention every time someone scrolls
through looking for what actually matters right now.

Knowing information exists is not the same as knowing it will actually get used well.
Where it sits, and what surrounds it, matters almost as much as whether it is there at all.